# recs_019 — V2a metadata Jaccard spike

**V2a rank-only rerank on frozen `two_tower_v1` @100 pools · train_tune → val**

Notebook-first spike for **IGDB FK-set Jaccard** (no taxonomy USE, no summary/storyline — those are V2b/V2c).

**Plan:** [`docs/recommender_v2_plan.md`](../../docs/recommender_v2_plan.md) · **D1 reference:** [`recs_013_ranker_d1_heuristic.ipynb`](recs_013_ranker_d1_heuristic.ipynb)

# Executive Summary

**Question:**  
Does IGDB taxonomy overlap (Jaccard) improve ranking @10 on frozen `two_tower_v1` pools vs D1 — including a **D1 + metadata** blend (`*_metadata_logpop_blend`)?

**Result:**  
Pure retr+meta Jaccard (**`two_tower_v1_v2a_query`**) still fails vs D1: **0.042** / **0.045** overall / Slice A vs D1 **0.093** / **0.068**. **`two_tower_v1_v2a_query_metadata_logpop_blend`** (D1 + Jaccard, `w_meta=0.05`, `genre_theme_kw`) **beats D1** on val: overall NDCG@10 **0.095** / Slice A **0.070** vs D1 **0.093** / **0.068**; personalization gap **0.725** vs D1 **0.720** (guardrail passes).

**Recommendation / Decision:**  
**Kill** retr+meta-only Jaccard for ship. **Promotion candidate:** `two_tower_v1_v2a_query_metadata_logpop_blend`. Compare head-to-head vs `recs_020` embed logpop_blend before eval-job wiring. History-anchor logpop_blend is close but below query on Slice A.

# Business Context

**Why are we doing this?**  
v1 shipped D1 (`two_tower_v1_heuristic_logpop_blend`). v2 adds interpretable IGDB metadata signals on the **same frozen retrieval pools** to beat D1 on val without changing retrieve.

**Expected Impact**  
If V2a lifts Slice A NDCG@10 without hurting personalization vs D1, we register `two_tower_v1_v2a_*` methods and wire `recs_job_eval_ranking`.

# Research Question

**Research Question:**  
On `train_ranker_v1` / `val_dev_12k_v1`, does weighted multi-field **Jaccard** reranking within frozen `two_tower_v1` pools beat bare retrieval, `popularity_train`, and D1 — for both **query-anchor** (query game tags) and **history-anchor** (union of train-like game tags)?

# Hypothesis

**Hypothesis:**  
Genre/theme/mode overlap between anchor tags and candidate tags reorders the frozen pool toward held-out positives better than popularity alone.

**Success Criteria (val, `k_final=10`):**
- Beat D1 on **Slice A** NDCG@10 (primary, same as recs_013)
- Do not worsen **PersonalizationGapVsPopularity@10** vs D1 (~0.72)
- Report IGDB coverage slice (apps with any taxonomy FK vs missing)

# Definitions

| Term | Definition | Notes |
|------|------------|-------|
| V2a | Metadata rerank via **Jaccard** on IGDB FK id sets | Not semantic USE on taxonomy; not `summary__use` (V2b) |
| V2a-query | Anchor = FK sets of `query_app_id` | Always available |
| V2a-history | Anchor = union of FK sets from **train-like** apps | From `eval_examples.parquet` `train_review_rows`; empty → falls back to query anchor |
| Field score | Per-field Jaccard(anchor_set, candidate_set) | Empty∪empty → 1.0; empty vs non-empty → 0.0 |
| Aggregate score | Weighted mean of active fields, min-max per pool | Same discipline as D1 pool normalization |
| train_tune | 10% stratified holdout of `train_ranker_v1` pools | Hyperparams chosen here only; val scored once |
| Slice A | `slice_a_multi_target` (`n_eval_targets >= 2`) | Primary tuning / promotion metric |
| D1 | `two_tower_v1_heuristic_logpop_blend`, α=0.2 | Fixed from recs_013 |

# Data Sources

## Train ranker pools

**Source Name:** Frozen `two_tower_v1` retrieval pools (train cohort)

**Location:** `artifacts/recs/ranker_pools/train_ranker_v1/two_tower_v1.parquet`

**Population Covered:** 51,691 examples from `train_ranker_v1` (disjoint from val)

**Filters Applied:** Built via `recs_job_export_retrieval_pools_train_ranker.json`; `retrieval_k=100`

**Known Limitations:** Cohort has `n_support_train=0` — no train_review_rows for history-anchor tuning on train

---

## Val pools + examples

**Source Name:** Val frozen pools + full example records (for history anchor)

**Location:**
- Pools: `artifacts/recs/offline_eval/runs/latest/eval_offline_examples.jsonl`
- Examples: `artifacts/recs/eval_cache/val_dev_12k_v1/eval_examples.parquet`

**Population Covered:** 12,500 val examples (`val_dev_12k_v1`)

**Filters Applied:** `min_review_chars=30`, `max_train_rows_per_user=5`, seed `2026`

**Known Limitations:** ~75% have train history; IGDB join covers 315-game catalog subset

---

## IGDB enriched metadata

**Source Name:** Job 2 enriched games (`*_names`, FK id columns)

**Location:** `artifacts/igdb/igdb_games__enriched.parquet`

**Population Covered:** 315 Steam apps in current catalog

**Filters Applied:** Job 1 + Job 2 pipeline (`recs_job_igdb_games.py`, `recs_job_igdb_games_enriched.py`)

**Known Limitations:** V2a uses FK columns only (`genres`, `themes`, `keywords`, `game_modes`, `player_perspectives`); apps missing from parquet get empty tag sets

# Design / Process

**Methodology**  
Offline rank-only eval on frozen pools. Hyperparameter search on **train_tune** (90/10 stratified by `slice_name`, seed `42`). Val scored once with fixed params — no val tuning.

**Analysis Approach**
1. Build per-app FK id sets from IGDB enriched parquet
2. Score each pool candidate: weighted mean of per-field Jaccards vs anchor
3. Grid-search field subsets + optional retrieval blend α on train_tune (query anchor; Slice A NDCG@10)
4. Val face-off: baselines + V2a-query + V2a-history @ tuned config
5. Emit full ranking metric tables (overall, slice, support, pop decile, personalization)

**Prerequisites (run once if artifacts missing):**
```bash
python scripts/recs_job_igdb_games.py configs/recs_job_igdb_games.json
python scripts/recs_job_igdb_games_enriched.py configs/recs_job_igdb_games_enriched.json
python scripts/recs_job_build_example_cohort.py configs/recs_job_build_example_cohort_train_ranker.json
python scripts/recs_job_export_retrieval_pools.py configs/recs_job_export_retrieval_pools_train_ranker.json
python scripts/recs_job_eval_offline.py configs/recs_job_eval_offline.json \
  --examples-parquet artifacts/recs/eval_cache/val_dev_12k_v1/eval_examples.parquet
```

# Decision Log

| Decision | Reason | Alternatives Considered | Impact |
|----------|--------|-------------------------|--------|
| Jaccard on FK ids only | V2a scope per plan + user sign-off | Taxonomy USE cosine, entity USE | Keeps spike interpretable; embeddings deferred |
| Weighted per-field mean | Tunable field importance readout | Single union-Jaccard across all tags | More knobs; matches EDA field coverage differences |
| Tune on query anchor for train | `train_ranker_v1` has zero train_review_rows | Skip tuning; tune on val | No leakage; history variant eval-only on val |
| History = union of train-like FK sets | Plan default V2a-history | Mean of per-app Jaccards | Broader user taste profile |
| Optional +retrieval blend α | Ablation vs pure metadata rerank | D1+meta `_logpop_blend` (this notebook) | Retr+meta grid on train_tune |
| D1 + metadata `w_meta` | Tests metadata on top of shipped D1 | Pure metadata or retr+meta only | Second grid; `w_meta=0` sanity = D1 |
| Exclude `franchises` | EDA: sparse / not in Job 2 resolve set | Include per v2 plan table | Five FK fields from `TAXONOMY_RESOLVE_FIELDS` |

# Evaluation Outputs / Artifacts

| File / Artifact Name | Location | Description | How it was generated | How it should be used or interpreted |
|----------------------|----------|-------------|------------------------|--------------------------------------|
| `v2a_train_tune_grid.csv` | `artifacts/recs/spikes/v2a/` | Field-subset × α grid on train_tune | Train-tune cell | Pick config; do not re-tune on val |
| `v2a_val_per_example.parquet` | `artifacts/recs/spikes/v2a/` | Per-example ranking metrics | Val face-off cell | Slice / support joins |
| `v2a_val_overall.csv` | `artifacts/recs/spikes/v2a/` | Method-level means | Aggregation helpers | Primary promotion table |
| `v2a_val_by_slice.csv` | `artifacts/recs/spikes/v2a/` | Metrics by slice | Aggregation helpers | Slice A gate |
| `v2a_val_personalization.csv` | `artifacts/recs/spikes/v2a/` | Personalization diagnostics | `_table_personalization` | Guardrail vs D1 |
| `v2a_plus_d1_train_tune_grid.csv` | `artifacts/recs/spikes/v2a/` | D1+meta `w_meta` grid | Plus-D1 train_tune cell | Pick `BEST_W_META` |

# Notebook Roadmap

1. Setup, paths, reproducibility constants
2. Load train/val pools, catalog, IGDB metadata, val examples (history)
3. Validate IGDB coverage on pool apps
4. Jaccard scoring + train_tune grid (field subsets, optional α blend)
5. D1 + metadata ablation (`_logpop_blend`): tune `w_meta` on train_tune
6. Val face-off vs baselines (two_tower, D1, pop, oracle, `*_logpop_blend`)
7. Full metric tables + save artifacts
8. Key findings + recommendation


# Analysis

_Add analysis code, visualizations, and intermediate results below. Keep cells focused and document non-obvious steps in markdown cells between code blocks._

## Setup

In [1]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Callable, Literal

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from steam_review_ml.evaluation.example_cohort import load_retrieval_pool_rows, load_retrieval_pools_jsonl
from steam_review_ml.evaluation.heuristic_ranker import (
    DEFAULT_LOGPOP_BLEND_ALPHA,
    minmax_norm,
    score_logpop_blend,
)
from steam_review_ml.evaluation.retrieval_offline_eval import (
    RANKING_REPORT_METRIC_COLS,
    _append_personalization_metrics,
    _oracle_ranked_indices_from_retrieved,
    _rank_rows,
    _table_by_slice_for_metrics,
    _table_by_support_for_metrics,
    _table_overall_ranking,
    _table_personalization,
    _table_popularity,
    average_precision_at_k,
    hit_rate_at_k,
    load_ranking_catalog_context,
    mrr,
    ndcg_at_k,
    precision_at_k,
    recall_at_k,
)
from steam_review_ml.igdb.constants import TAXONOMY_RESOLVE_FIELDS
from steam_review_ml.recommender.retrieve import ContentRetriever

# --- Replicability form ---
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
RANDOM_SEED = 42
SPLIT_SEED = 42
TUNE_FRAC = 0.10
K_FINAL = 10
K_PERSONALIZATION = 10
MIN_REVIEW_CHARS = 30
POOL_METHOD = "two_tower_v1"
D1_ALPHA = DEFAULT_LOGPOP_BLEND_ALPHA

TRAIN_POOLS_PARQUET = REPO_ROOT / "artifacts/recs/ranker_pools/train_ranker_v1/two_tower_v1.parquet"
VAL_JSONL = REPO_ROOT / "artifacts/recs/offline_eval/runs/latest/eval_offline_examples.jsonl"
VAL_EXAMPLES_PARQUET = REPO_ROOT / "artifacts/recs/eval_cache/val_dev_12k_v1/eval_examples.parquet"
IGDB_ENRICHED = REPO_ROOT / "artifacts/igdb/igdb_games__enriched.parquet"
ARTIFACT_DIR = REPO_ROOT / "artifacts/recs"
SPIKE_OUT = ARTIFACT_DIR / "spikes/v2a"

TAXONOMY_FIELDS: tuple[str, ...] = TAXONOMY_RESOLVE_FIELDS

FIELD_PRESETS: dict[str, tuple[str, ...]] = {
    "all5": TAXONOMY_FIELDS,
    "no_keywords": ("genres", "themes", "game_modes", "player_perspectives"),
    "genre_theme": ("genres", "themes"),
    "genre_theme_mode": ("genres", "themes", "game_modes"),
    "genre_theme_kw": ("genres", "themes", "keywords"),
}

BLEND_ALPHAS = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
PLUS_D1_WEIGHTS = [0.0, 0.05, 0.1, 0.15, 0.2, 0.3]  # w on metadata; 0 = D1 only  # 1.0 = retrieval only; 0.0 = metadata only

for p in (TRAIN_POOLS_PARQUET, VAL_JSONL, VAL_EXAMPLES_PARQUET, IGDB_ENRICHED):
    if not p.is_file():
        raise FileNotFoundError(f"Missing required artifact: {p}")

SPIKE_OUT.mkdir(parents=True, exist_ok=True)
print(f"REPO_ROOT={REPO_ROOT}")
print(f"SPIKE_OUT={SPIKE_OUT}")

/home/ryanr/miniconda3/envs/tf_condaforge/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-06-22 14:37:14.688373: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-22 14:37:14.774031: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782153434.798738 3536666 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782153434.808305 3536666 cuda_bl

REPO_ROOT=/home/ryanr/workspace/steam_recommendations
SPIKE_OUT=/home/ryanr/workspace/steam_recommendations/artifacts/recs/spikes/v2a


In [2]:
#no truncation of pandas columns
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:.5f}'.format)

## Load Data

In [3]:
train_pools = load_retrieval_pool_rows(TRAIN_POOLS_PARQUET)
val_pools = load_retrieval_pools_jsonl(VAL_JSONL, method=POOL_METHOD)
val_pools_by_ex = {int(r["ex_idx"]): r for r in val_pools}

catalog = load_ranking_catalog_context(
    repo_root=REPO_ROOT,
    min_review_chars=MIN_REVIEW_CHARS,
    artifact_dir=ARTIFACT_DIR,
)
app_ids = catalog.app_ids
app_to_row = catalog.app_to_row
pop_row = catalog.pop_row

retriever = ContentRetriever(artifact_dir=ARTIFACT_DIR, repo_root=REPO_ROOT)
X_emb = retriever.embedding_matrix

val_examples_df = pd.read_parquet(VAL_EXAMPLES_PARQUET)
history_apps_by_ex: dict[int, set[int]] = {}
for _, row in val_examples_df.iterrows():
    ex_idx = int(row["ex_idx"])
    train_rows = json.loads(row["train_review_rows_json"])
    history_apps_by_ex[ex_idx] = {int(r["app_id"]) for r in train_rows}

examples_for_pers: list[dict[str, Any]] = []
for _, row in val_examples_df.iterrows():
    examples_for_pers.append(
        {
            "ex_idx": int(row["ex_idx"]),
            "user_id": row["user_id"],
            "query_app_id": int(row["query_app_id"]),
            "query_ts": float(row["query_ts"]),
            "n_eval_targets": int(row["n_eval_targets"]),
            "train_review_rows": json.loads(row["train_review_rows_json"]),
            "validation_positive_app_ids": json.loads(row["validation_positive_app_ids_json"]),
        }
    )

support_by_ex = val_examples_df.set_index("ex_idx")["n_support_train"].astype(int).to_dict()
for row in val_pools:
    row["n_support_train"] = int(support_by_ex.get(int(row["ex_idx"]), 0))

print(
    f"train pools: {len(train_pools):,}  val pools: {len(val_pools):,}  "
    f"catalog apps: {len(app_ids):,}  val w/ history: {sum(1 for s in history_apps_by_ex.values() if s):,}"
)

train pools: 51,691  val pools: 12,500  catalog apps: 315  val w/ history: 9,375


## Validate Data Quality

In [4]:
igdb_df = pd.read_parquet(IGDB_ENRICHED, columns=["app_id", *TAXONOMY_FIELDS])


def parse_fk_set(val: Any) -> frozenset[int]:
    """Normalize one IGDB FK column value to a set of integer ids.

    Parquet cells may be ``None``, NaN, or a list/array of ids. Used when building
    ``field_sets_by_app`` for fast Jaccard lookups during scoring.
    """
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return frozenset()
    if isinstance(val, (list, tuple, np.ndarray)):
        return frozenset(int(x) for x in val)
    return frozenset()


# Lookup index: app_id -> {field -> frozenset of IGDB FK ids}
# Built once here; used by all Jaccard scoring below.
field_sets_by_app: dict[int, dict[str, frozenset[int]]] = {}
for _, row in igdb_df.iterrows():
    app_id = int(row["app_id"])
    field_sets_by_app[app_id] = {f: parse_fk_set(row[f]) for f in TAXONOMY_FIELDS}

catalog_app_set = {int(a) for a in app_ids}
igdb_app_set = set(field_sets_by_app)
pool_apps: set[int] = set()
for pools in (train_pools, val_pools):
    for row in pools:
        pool_apps.update(int(x) for x in json.loads(row["retrieved_app_ids_json"]))
        pool_apps.add(int(row["query_app_id"]))

cov = {
    "catalog_apps": len(catalog_app_set),
    "igdb_apps": len(igdb_app_set),
    "pool_unique_apps": len(pool_apps),
    "pool_in_igdb": len(pool_apps & igdb_app_set),
    "pool_missing_igdb": len(pool_apps - igdb_app_set),
}
display(pd.Series(cov, name="igdb_coverage"))

per_field_nonempty = {}
for f in TAXONOMY_FIELDS:
    per_field_nonempty[f] = sum(1 for a in catalog_app_set if field_sets_by_app.get(a, {}).get(f))
display(pd.Series(per_field_nonempty, name="catalog_apps_with_field"))

catalog_apps         315
igdb_apps            315
pool_unique_apps     315
pool_in_igdb         315
pool_missing_igdb      0
Name: igdb_coverage, dtype: int64

genres                 315
themes                 310
keywords               287
game_modes             315
player_perspectives    302
Name: catalog_apps_with_field, dtype: int64

## Feature Engineering / Preprocessing

In [5]:
AnchorMode = Literal["query", "history"]  # query = V2a-query; history = V2a-history

# --- V2a scoring primitives (Jaccard on FK sets, pool-level rerank) ---


def jaccard(a: frozenset[int], b: frozenset[int]) -> float:
    """Jaccard similarity on two FK id sets.

    Returns ``|A ∩ B| / |A U B|``. Both empty → 1.0 (no discriminating signal).
    One empty → 0.0.
    """
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)


def anchor_field_sets(
    *,
    query_app_id: int,
    history_app_ids: set[int],
    mode: AnchorMode,
) -> dict[str, frozenset[int]]:
    """Build per-field anchor tag sets for one example.

    - ``query``: tags from ``query_app_id`` only (V2a-query).
    - ``history``: union of tags across ``history_app_ids`` (V2a-history).
      If history is empty, falls back to query anchor so scoring still runs.
    """
    if mode == "query" or not history_app_ids:
        src_apps = [int(query_app_id)]
    else:
        src_apps = sorted(history_app_ids)
    out = {f: frozenset() for f in TAXONOMY_FIELDS}
    for app_id in src_apps:
        per_app = field_sets_by_app.get(int(app_id), {})
        for f in TAXONOMY_FIELDS:
            out[f] |= per_app.get(f, frozenset())
    return out


def metadata_pool_scores(
    pool_app_ids: list[int],
    retrieval_scores: list[float] | np.ndarray,
    *,
    fields: tuple[str, ...],
    field_weights: dict[str, float] | None = None,
    alpha: float,
    query_app_id: int,
    history_app_ids: set[int],
    anchor_mode: AnchorMode,
) -> np.ndarray:
    """Score every app in a frozen retrieval pool for reranking.

    Steps per pool:
    1. Jaccard(anchor, candidate) per active field → weighted mean → ``meta``
    2. Min-max normalize ``meta`` and retrieval scores within the pool
    3. Return ``alpha * norm(retr) + (1 - alpha) * norm(meta)``

    ``alpha=0`` is pure metadata; ``alpha=1`` is pure two-tower (no metadata effect).
    """
    weights = field_weights or {f: 1.0 for f in fields}
    anchor = anchor_field_sets(
        query_app_id=query_app_id,
        history_app_ids=history_app_ids,
        mode=anchor_mode,
    )
    meta = np.empty(len(pool_app_ids), dtype=np.float64)
    for i, app_id in enumerate(pool_app_ids):
        cand = field_sets_by_app.get(int(app_id), {})
        num = 0.0
        den = 0.0
        for f in fields:
            w = float(weights.get(f, 1.0))
            num += w * jaccard(anchor[f], cand.get(f, frozenset()))
            den += w
        meta[i] = num / den if den > 0 else 0.0
    retr = minmax_norm(np.asarray(retrieval_scores, dtype=np.float64))
    meta_n = minmax_norm(meta)
    return alpha * retr + (1.0 - alpha) * meta_n

def raw_metadata_pool_scores(
    pool_app_ids: list[int],
    *,
    fields: tuple[str, ...],
    field_weights: dict[str, float] | None = None,
    query_app_id: int,
    history_app_ids: set[int],
    anchor_mode: AnchorMode,
) -> np.ndarray:
    """Pure metadata signal (no retrieval blend) for D1+meta ablation."""
    weights = field_weights or {f: 1.0 for f in fields}
    anchor = anchor_field_sets(
        query_app_id=query_app_id,
        history_app_ids=history_app_ids,
        mode=anchor_mode,
    )
    meta = np.empty(len(pool_app_ids), dtype=np.float64)
    for i, app_id in enumerate(pool_app_ids):
        cand = field_sets_by_app.get(int(app_id), {})
        num = 0.0
        den = 0.0
        for f in fields:
            w = float(weights.get(f, 1.0))
            num += w * jaccard(anchor[f], cand.get(f, frozenset()))
            den += w
        meta[i] = num / den if den > 0 else 0.0
    return meta


def d1_plus_metadata_pool_scores(
    pool_app_ids: list[int],
    retrieval_scores: list[float] | np.ndarray,
    *,
    w_meta: float,
    fields: tuple[str, ...],
    field_weights: dict[str, float] | None = None,
    query_app_id: int,
    history_app_ids: set[int],
    anchor_mode: AnchorMode,
) -> np.ndarray:
    """Convex mix: ``(1 - w_meta) * norm(D1) + w_meta * norm(metadata)``.

    ``w_meta=0`` returns D1 unchanged (sanity check vs shipped ranker).
    """
    d1 = score_logpop_blend(
        pool_app_ids,
        retrieval_scores,
        alpha=D1_ALPHA,
        pop_row=pop_row,
        app_to_row=app_to_row,
    )
    meta = raw_metadata_pool_scores(
        pool_app_ids,
        fields=fields,
        field_weights=field_weights,
        query_app_id=query_app_id,
        history_app_ids=history_app_ids,
        anchor_mode=anchor_mode,
    )
    if w_meta <= 0.0:
        return d1
    if w_meta >= 1.0:
        return minmax_norm(meta)
    return (1.0 - w_meta) * minmax_norm(d1) + w_meta * minmax_norm(meta)



def pool_scores_to_ranked_indices(
    pool_app_ids: list[int],
    pool_scores: np.ndarray,
    *,
    k_final: int,
) -> np.ndarray:
    """Map per-pool scores to top-``k_final`` catalog row indices.

    Writes scores into a full-catalog vector (non-pool apps stay ``-inf``), then
    argsorts descending — same contract as recs_013 pool rerank helpers.
    """
    full = np.full(len(app_ids), -np.inf, dtype=np.float64)
    for app_id, score in zip(pool_app_ids, pool_scores):
        full[int(app_to_row[int(app_id)])] = float(score)
    return _rank_rows(full)[:k_final]


def stratified_ex_idx_split(
    pools: list[dict[str, Any]], *, tune_frac: float, seed: int
) -> tuple[set[int], set[int]]:
    """Split example indices into fit vs tune, stratified by ``slice_name``.

    Within each slice, shuffles ``ex_idx`` and allocates ~``tune_frac`` to tune
    (at least one example per slice). Returns ``(fit_ex_idx, tune_ex_idx)``.
    """
    rng = np.random.default_rng(seed)
    by_slice: dict[str, list[int]] = {}
    for row in pools:
        by_slice.setdefault(str(row["slice_name"]), []).append(int(row["ex_idx"]))
    fit_ids: set[int] = set()
    tune_ids: set[int] = set()
    for ids in by_slice.values():
        ids_arr = np.asarray(sorted(ids))
        rng.shuffle(ids_arr)
        n_tune = max(1, int(round(len(ids_arr) * tune_frac)))
        tune_ids.update(int(x) for x in ids_arr[:n_tune])
        fit_ids.update(int(x) for x in ids_arr[n_tune:])
    return fit_ids, tune_ids


_, tune_ex_idx = stratified_ex_idx_split(train_pools, tune_frac=TUNE_FRAC, seed=SPLIT_SEED)
train_tune = [r for r in train_pools if int(r["ex_idx"]) in tune_ex_idx]
print(f"train_tune={len(train_tune):,} / {len(train_pools):,}")

train_tune=5,169 / 51,691


## Core Analysis

In [6]:
def mean_ndcg_slice_a(
    pools: list[dict[str, Any]],
    *,
    fields: tuple[str, ...],
    alpha: float,
    anchor_mode: AnchorMode = "query",
) -> float:
    """Mean NDCG@K on Slice A examples only — used for train_tune hyperparameter search.

    Skips examples with ``n_eval_targets < 2`` or no validation positives. Scores each
    pool with ``metadata_pool_scores``, reranks @K, and averages ``ndcg_at_k``.
    """
    vals: list[float] = []
    for row in pools:
        if int(row["n_eval_targets"]) < 2:
            continue
        positives = set(int(x) for x in json.loads(row["validation_positive_app_ids_json"]))
        if not positives:
            continue
        pool_apps = [int(x) for x in json.loads(row["retrieved_app_ids_json"])]
        ret_sc = json.loads(row["retrieved_scores_json"])
        ex_idx = int(row["ex_idx"])
        history = history_apps_by_ex.get(ex_idx, set()) if anchor_mode == "history" else set()
        blend = metadata_pool_scores(
            pool_apps,
            ret_sc,
            fields=fields,
            alpha=alpha,
            query_app_id=int(row["query_app_id"]),
            history_app_ids=history,
            anchor_mode=anchor_mode,
        )
        ranked = pool_scores_to_ranked_indices(pool_apps, blend, k_final=K_FINAL)
        vals.append(ndcg_at_k(ranked, positives, K_FINAL, app_ids))
    return float(np.mean(vals)) if vals else float("nan")


grid_rows: list[dict[str, Any]] = []
for preset_name, fields in FIELD_PRESETS.items():
    for alpha in BLEND_ALPHAS:
        grid_rows.append(
            {
                "field_preset": preset_name,
                "fields": ",".join(fields),
                "alpha": alpha,
                "train_tune_NDCG_slice_a_query": mean_ndcg_slice_a(
                    train_tune, fields=fields, alpha=alpha, anchor_mode="query"
                ),
            }
        )

train_grid = pd.DataFrame(grid_rows).sort_values("train_tune_NDCG_slice_a_query", ascending=False)
best = train_grid.iloc[0]
BEST_FIELDS = tuple(FIELD_PRESETS[str(best["field_preset"])])
BEST_ALPHA = float(best["alpha"])
BEST_PRESET = str(best["field_preset"])

display(Markdown("### Train_tune grid (query anchor · Slice A NDCG@10)"))
display(train_grid.head(12))
print(f"BEST: preset={BEST_PRESET} alpha={BEST_ALPHA} fields={BEST_FIELDS}")

train_grid.to_csv(SPIKE_OUT / "v2a_train_tune_grid.csv", index=False)

### Train_tune grid (query anchor · Slice A NDCG@10)

,field_preset,fields,alpha,train_tune_NDCG_slice_a_query
24,genre_theme_kw,"genres,themes,keywords",0.00000,0.06949
0,all5,"genres,themes,keywords,game_modes,player_perspectives",0.00000,0.06931
25,genre_theme_kw,"genres,themes,keywords",0.20000,0.06744
1,all5,"genres,themes,keywords,game_modes,player_perspectives",0.20000,0.06695
6,no_keywords,"genres,themes,game_modes,player_perspectives",0.00000,0.06660
12,genre_theme,"genres,themes",0.00000,0.06466
7,no_keywords,"genres,themes,game_modes,player_perspectives",0.20000,0.06432
13,genre_theme,"genres,themes",0.20000,0.06360
18,genre_theme_mode,"genres,themes,game_modes",0.00000,0.06340
19,genre_theme_mode,"genres,themes,game_modes",0.20000,0.06183


BEST: preset=genre_theme_kw alpha=0.0 fields=('genres', 'themes', 'keywords')


## D1 + metadata ablation (`_logpop_blend`)

Prior grid blended **retrieval + metadata** (`α·retr + (1−α)·meta`). D1 is **`0.2·retr + 0.8·log_pop`** — a different signal. This section tunes **`w_meta`** on train_tune for:

```text
score = (1 − w_meta) · norm(D1) + w_meta · norm(metadata)
```

Metadata hyperparams fixed from the grid above (`BEST_FIELDS`, etc.). Tune on **query** anchor only; report val for query + history variants.


In [7]:
def mean_ndcg_slice_a_plus_d1(
    pools: list[dict[str, Any]],
    *,
    fields: tuple[str, ...],
    w_meta: float,
    anchor_mode: AnchorMode = "query",
) -> float:
    """Mean Slice A NDCG@K for D1+metadata blend — train_tune search only."""
    vals: list[float] = []
    for row in pools:
        if int(row["n_eval_targets"]) < 2:
            continue
        positives = set(int(x) for x in json.loads(row["validation_positive_app_ids_json"]))
        if not positives:
            continue
        pool_apps = [int(x) for x in json.loads(row["retrieved_app_ids_json"])]
        ret_sc = json.loads(row["retrieved_scores_json"])
        ex_idx = int(row["ex_idx"])
        history = history_apps_by_ex.get(ex_idx, set()) if anchor_mode == "history" else set()
        blend = d1_plus_metadata_pool_scores(
            pool_apps,
            ret_sc,
            w_meta=w_meta,
            fields=fields,
            query_app_id=int(row["query_app_id"]),
            history_app_ids=history,
            anchor_mode=anchor_mode,
        )
        ranked = pool_scores_to_ranked_indices(pool_apps, blend, k_final=K_FINAL)
        vals.append(ndcg_at_k(ranked, positives, K_FINAL, app_ids))
    return float(np.mean(vals)) if vals else float("nan")


plus_d1_grid_rows: list[dict[str, Any]] = []
for w_meta in PLUS_D1_WEIGHTS:
    plus_d1_grid_rows.append(
        {
            "w_meta": w_meta,
            "train_tune_NDCG_slice_a_query": mean_ndcg_slice_a_plus_d1(
                train_tune, fields=BEST_FIELDS, w_meta=w_meta, anchor_mode="query"
            ),
        }
    )

plus_d1_train_grid = pd.DataFrame(plus_d1_grid_rows).sort_values(
    "train_tune_NDCG_slice_a_query", ascending=False
)
plus_d1_best = plus_d1_train_grid.iloc[0]
BEST_W_META = float(plus_d1_best["w_meta"])

display(Markdown("### Train_tune grid — D1 + metadata (query anchor · Slice A NDCG@10)"))
display(plus_d1_train_grid)
print(f"BEST_PLUS_D1: w_meta={BEST_W_META} fields={BEST_FIELDS}")

plus_d1_train_grid.to_csv(SPIKE_OUT / "v2a_plus_d1_train_tune_grid.csv", index=False)


### Train_tune grid — D1 + metadata (query anchor · Slice A NDCG@10)

,w_meta,train_tune_NDCG_slice_a_query
1,0.05000,0.17173
2,0.10000,0.17125
0,0.00000,0.17012
3,0.15000,0.16938
4,0.20000,0.16527
5,0.30000,0.15287


BEST_PLUS_D1: w_meta=0.05 fields=('genres', 'themes', 'keywords')


In [8]:
train_grid

,field_preset,fields,alpha,train_tune_NDCG_slice_a_query
24,genre_theme_kw,"genres,themes,keywords",0.00000,0.06949
0,all5,"genres,themes,keywords,game_modes,player_perspectives",0.00000,0.06931
25,genre_theme_kw,"genres,themes,keywords",0.20000,0.06744
1,all5,"genres,themes,keywords,game_modes,player_perspectives",0.20000,0.06695
6,no_keywords,"genres,themes,game_modes,player_perspectives",0.00000,0.06660
12,genre_theme,"genres,themes",0.00000,0.06466
7,no_keywords,"genres,themes,game_modes,player_perspectives",0.20000,0.06432
13,genre_theme,"genres,themes",0.20000,0.06360
18,genre_theme_mode,"genres,themes,game_modes",0.00000,0.06340
19,genre_theme_mode,"genres,themes,game_modes",0.20000,0.06183


## Evaluation

In [9]:
def per_example_ranking_row(
    row: dict[str, Any],
    *,
    method: str,
    score_fn: Callable[..., np.ndarray] | None = None,
    score_kwargs: dict[str, Any] | None = None,
    oracle: bool = False,
    catalog_pop: bool = False,
) -> dict[str, Any] | None:
    """Ranking metrics for one val example and one method.

    Modes:
    - ``score_fn=None``: raw frozen retrieval scores
    - ``score_fn`` set: pool rerank via custom scorer (V2a, D1, etc.)
    - ``oracle=True``: perfect reorder within the frozen top-100 pool
    - ``catalog_pop=True``: full-catalog ``popularity_train`` baseline

    Always attaches ``OracleHit@K`` / ``OracleNDCG@K`` from the frozen pool so
    aggregation helpers match the ``ranking_offline_eval`` contract.
    """
    positives = set(int(x) for x in json.loads(row["validation_positive_app_ids_json"]))
    if not positives:
        return None

    pool_apps = [int(x) for x in json.loads(row["retrieved_app_ids_json"])]
    ret_sc = [float(x) for x in json.loads(row["retrieved_scores_json"])]
    retrieved_rows = np.asarray([app_to_row[a] for a in pool_apps], dtype=np.int64)
    oracle_indices = _oracle_ranked_indices_from_retrieved(retrieved_rows, positives, app_ids)

    if oracle:
        ranked = oracle_indices[:K_FINAL]
    elif catalog_pop:
        s = np.asarray(pop_row, dtype=np.float64).copy()
        qrow = app_to_row.get(int(row["query_app_id"]))
        if qrow is not None:
            s[qrow] = -np.inf
        ranked = _rank_rows(s)[:K_FINAL]
    elif score_fn is None:
        ranked = pool_scores_to_ranked_indices(pool_apps, np.asarray(ret_sc), k_final=K_FINAL)
    else:
        blend = score_fn(pool_apps, ret_sc, **(score_kwargs or {}))
        ranked = pool_scores_to_ranked_indices(pool_apps, blend, k_final=K_FINAL)

    return {
        "method": method,
        "ex_idx": int(row["ex_idx"]),
        "slice_name": row["slice_name"],
        "n_eval_targets": int(row["n_eval_targets"]),
        "n_support_train": int(row.get("n_support_train", 0)),
        "query_app_id": int(row["query_app_id"]),
        "Hit@K": hit_rate_at_k(ranked, positives, K_FINAL, app_ids),
        "Precision@K": precision_at_k(ranked, positives, K_FINAL, app_ids),
        "Recall@K": recall_at_k(ranked, positives, K_FINAL, app_ids),
        "MAP@K": average_precision_at_k(ranked, positives, K_FINAL, app_ids),
        "NDCG@K": ndcg_at_k(ranked, positives, K_FINAL, app_ids),
        "MRR": mrr(ranked, positives, app_ids),
        "OracleHit@K": hit_rate_at_k(oracle_indices, positives, K_FINAL, app_ids),
        "OracleNDCG@K": ndcg_at_k(oracle_indices, positives, K_FINAL, app_ids),
    }


def make_v2a_score_fn(anchor_mode: AnchorMode, ex_idx: int, query_app_id: int) -> Callable[..., np.ndarray]:
    """Closure that scores a pool with the train-tuned V2a config for one example."""

    def _score(pool_apps: list[int], ret_sc: list[float]) -> np.ndarray:
        return metadata_pool_scores(
            pool_apps,
            ret_sc,
            fields=BEST_FIELDS,
            alpha=BEST_ALPHA,
            query_app_id=query_app_id,
            history_app_ids=history_apps_by_ex.get(ex_idx, set()),
            anchor_mode=anchor_mode,
        )

    return _score


def make_v2a_plus_d1_score_fn(anchor_mode: AnchorMode, ex_idx: int, query_app_id: int) -> Callable[..., np.ndarray]:
    """D1 + metadata blend at train-tuned ``BEST_W_META``."""

    def _score(pool_apps: list[int], ret_sc: list[float]) -> np.ndarray:
        return d1_plus_metadata_pool_scores(
            pool_apps,
            ret_sc,
            w_meta=BEST_W_META,
            fields=BEST_FIELDS,
            query_app_id=query_app_id,
            history_app_ids=history_apps_by_ex.get(ex_idx, set()),
            anchor_mode=anchor_mode,
        )

    return _score


val_metric_rows: list[dict[str, Any]] = []
for row in val_pools:
    ex_idx = int(row["ex_idx"])
    qid = int(row["query_app_id"])

    for method, kwargs in (
        (POOL_METHOD, {}),
        (
            "two_tower_v1_heuristic_logpop_blend",
            {
                "score_fn": score_logpop_blend,
                "score_kwargs": {"alpha": D1_ALPHA, "pop_row": pop_row, "app_to_row": app_to_row},
            },
        ),
        ("popularity_train", {"catalog_pop": True}),
        (f"{POOL_METHOD}_oracle", {"oracle": True}),
        (
            "two_tower_v1_v2a_query",
            {"score_fn": make_v2a_score_fn("query", ex_idx, qid)},
        ),
        (
            "two_tower_v1_v2a_history",
            {"score_fn": make_v2a_score_fn("history", ex_idx, qid)},
        ),
        (
            "two_tower_v1_v2a_query_metadata_logpop_blend",
            {"score_fn": make_v2a_plus_d1_score_fn("query", ex_idx, qid)},
        ),
        (
            "two_tower_v1_v2a_history_metadata_logpop_blend",
            {"score_fn": make_v2a_plus_d1_score_fn("history", ex_idx, qid)},
        ),
    ):
        mrow = per_example_ranking_row(row, method=method, **kwargs)
        if mrow:
            val_metric_rows.append(mrow)

df_val = pd.DataFrame(val_metric_rows)
df_val.to_parquet(SPIKE_OUT / "v2a_val_per_example.parquet", index=False)

overall = _table_overall_ranking(df_val)
by_slice = _table_by_slice_for_metrics(df_val, metric_cols=RANKING_REPORT_METRIC_COLS, ranking=True)
by_support = _table_by_support_for_metrics(df_val, metric_cols=RANKING_REPORT_METRIC_COLS, ranking=True)

pop_table, pop_delta, _ = _table_popularity(
    df_ex_metrics=df_val,
    examples=examples_for_pers,
    app_ids=app_ids,
    pop_row=pop_row,
    enable_popularity_decile_diagnostics=True,
    metric_cols=RANKING_REPORT_METRIC_COLS,
)

overall.to_csv(SPIKE_OUT / "v2a_val_overall.csv", index=False)
by_slice.to_csv(SPIKE_OUT / "v2a_val_by_slice.csv", index=False)
by_support.to_csv(SPIKE_OUT / "v2a_val_by_support.csv", index=False)
pop_table.to_csv(SPIKE_OUT / "v2a_val_by_pop_decile.csv", index=False)
pop_delta.to_csv(SPIKE_OUT / "v2a_val_pop_delta.csv", index=False)

display(Markdown("### Val overall"))
display(overall.sort_values("NDCG@K", ascending=False))
display(Markdown("### Val by slice"))
display(by_slice.sort_values(["slice_name", "NDCG@K"], ascending=[True, False]))

### Val overall

,method,Hit@K,Precision@K,Recall@K,MAP@K,NDCG@K,MRR,OracleHit@K,OracleNDCG@K
0,two_tower_v1_oracle,0.51224,0.05414,0.49405,0.49405,0.49831,0.51224,0.51224,0.49831
1,two_tower_v1_v2a_query_metadata_logpop_blend,0.19664,0.01991,0.18743,0.06629,0.09518,0.06911,0.51224,0.49831
2,two_tower_v1_v2a_history_metadata_logpop_blend,0.19560,0.01980,0.18669,0.06385,0.09308,0.06652,0.51224,0.49831
3,two_tower_v1_heuristic_logpop_blend,0.19328,0.01956,0.18410,0.06432,0.09289,0.06706,0.51224,0.49831
4,popularity_train,0.15112,0.01518,0.14680,0.05059,0.07311,0.05222,0.51224,0.49831
5,two_tower_v1_v2a_query,0.08552,0.00859,0.07906,0.02995,0.04189,0.03201,0.51224,0.49831
6,two_tower_v1_v2a_history,0.08520,0.00855,0.07990,0.02494,0.03812,0.02650,0.51224,0.49831
7,two_tower_v1,0.04680,0.00473,0.04374,0.01032,0.01816,0.01101,0.51224,0.49831


### Val by slice

,slice_name,method,Hit@K,Precision@K,Recall@K,MAP@K,NDCG@K,MRR,OracleHit@K,OracleNDCG@K
0,slice_a_multi_target,two_tower_v1_oracle,0.77379,0.12772,0.46023,0.46023,0.53362,0.77379,0.77379,0.53362
1,slice_a_multi_target,two_tower_v1_v2a_query_metadata_logpop_blend,0.27310,0.03159,0.11434,0.03537,0.06986,0.08405,0.77379,0.53362
2,slice_a_multi_target,two_tower_v1_heuristic_logpop_blend,0.27172,0.03117,0.11337,0.03418,0.06832,0.08140,0.77379,0.53362
3,slice_a_multi_target,two_tower_v1_v2a_history_metadata_logpop_blend,0.26621,0.03076,0.11262,0.03324,0.06708,0.07921,0.77379,0.53362
4,slice_a_multi_target,two_tower_v1_v2a_query,0.17931,0.01862,0.06798,0.02369,0.04472,0.05920,0.77379,0.53362
5,slice_a_multi_target,popularity_train,0.12828,0.01407,0.05373,0.01942,0.03544,0.04747,0.77379,0.53362
6,slice_a_multi_target,two_tower_v1_v2a_history,0.14759,0.01531,0.05620,0.01597,0.03388,0.04278,0.77379,0.53362
7,slice_a_multi_target,two_tower_v1,0.09103,0.00993,0.03828,0.00941,0.02054,0.02119,0.77379,0.53362
8,slice_b_single_target,two_tower_v1_oracle,0.49614,0.04961,0.49614,0.49614,0.49614,0.49614,0.49614,0.49614
9,slice_b_single_target,two_tower_v1_v2a_query_metadata_logpop_blend,0.19193,0.01919,0.19193,0.06820,0.09673,0.06820,0.49614,0.49614


## Visualizations

In [10]:
def popularity_train_score(ex: dict[str, Any]) -> np.ndarray:
    s = np.asarray(pop_row, dtype=np.float64).copy()
    row = app_to_row.get(int(ex["query_app_id"]))
    if row is not None:
        s[row] = -np.inf
    return s.astype(np.float32)


def frozen_pool_score(ex: dict[str, Any]) -> np.ndarray:
    """Full-catalog scores from frozen ``two_tower_v1`` retrieval order (no rerank)."""
    row = val_pools_by_ex[int(ex["ex_idx"])]
    pool_apps = [int(x) for x in json.loads(row["retrieved_app_ids_json"])]
    ret_sc = [float(x) for x in json.loads(row["retrieved_scores_json"])]
    full = np.full(len(app_ids), -np.inf, dtype=np.float64)
    for app_id, score in zip(pool_apps, ret_sc):
        full[int(app_to_row[int(app_id)])] = score
    return full.astype(np.float32)


def make_pers_score_fn(anchor_mode: AnchorMode) -> Callable[[dict[str, Any]], np.ndarray]:
    """Build a full-catalog scorer for personalization diagnostics (V2a @ tuned config)."""

    def score(ex: dict[str, Any]) -> np.ndarray:
        row = val_pools_by_ex[int(ex["ex_idx"])]
        pool_apps = [int(x) for x in json.loads(row["retrieved_app_ids_json"])]
        ret_sc = [float(x) for x in json.loads(row["retrieved_scores_json"])]
        blend = metadata_pool_scores(
            pool_apps,
            ret_sc,
            fields=BEST_FIELDS,
            alpha=BEST_ALPHA,
            query_app_id=int(ex["query_app_id"]),
            history_app_ids=history_apps_by_ex.get(int(ex["ex_idx"]), set()),
            anchor_mode=anchor_mode,
        )
        full = np.full(len(app_ids), -np.inf, dtype=np.float64)
        for app_id, sc in zip(pool_apps, blend):
            full[int(app_to_row[int(app_id)])] = float(sc)
        return full.astype(np.float32)

    return score


def make_pers_plus_d1_score_fn(anchor_mode: AnchorMode) -> Callable[[dict[str, Any]], np.ndarray]:
    """Full-catalog scorer for D1+metadata personalization diagnostics."""

    def score(ex: dict[str, Any]) -> np.ndarray:
        row = val_pools_by_ex[int(ex["ex_idx"])]
        pool_apps = [int(x) for x in json.loads(row["retrieved_app_ids_json"])]
        ret_sc = [float(x) for x in json.loads(row["retrieved_scores_json"])]
        blend = d1_plus_metadata_pool_scores(
            pool_apps,
            ret_sc,
            w_meta=BEST_W_META,
            fields=BEST_FIELDS,
            query_app_id=int(ex["query_app_id"]),
            history_app_ids=history_apps_by_ex.get(int(ex["ex_idx"]), set()),
            anchor_mode=anchor_mode,
        )
        full = np.full(len(app_ids), -np.inf, dtype=np.float64)
        for app_id, sc in zip(pool_apps, blend):
            full[int(app_to_row[int(app_id)])] = float(sc)
        return full.astype(np.float32)

    return score


def d1_frozen_score(ex: dict[str, Any]) -> np.ndarray:
    """Full-catalog scores from D1 log-pop blend on the frozen pool (α fixed from recs_013)."""
    row = val_pools_by_ex[int(ex["ex_idx"])]
    pool_apps = [int(x) for x in json.loads(row["retrieved_app_ids_json"])]
    ret_sc = [float(x) for x in json.loads(row["retrieved_scores_json"])]
    blend = score_logpop_blend(
        pool_apps, ret_sc, alpha=D1_ALPHA, pop_row=pop_row, app_to_row=app_to_row
    )
    full = np.full(len(app_ids), -np.inf, dtype=np.float64)
    for app_id, sc in zip(pool_apps, blend):
        full[int(app_to_row[int(app_id)])] = float(sc)
    return full.astype(np.float32)


pers_methods: dict[str, Callable[[dict[str, Any]], np.ndarray]] = {
    "popularity_train": popularity_train_score,
    POOL_METHOD: frozen_pool_score,
    "two_tower_v1_heuristic_logpop_blend": d1_frozen_score,
    "two_tower_v1_v2a_query": make_pers_score_fn("query"),
    "two_tower_v1_v2a_history": make_pers_score_fn("history"),
    "two_tower_v1_v2a_query_metadata_logpop_blend": make_pers_plus_d1_score_fn("query"),
    "two_tower_v1_v2a_history_metadata_logpop_blend": make_pers_plus_d1_score_fn("history"),
}

personalization = _table_personalization(
    methods=pers_methods,
    examples=examples_for_pers,
    X=X_emb,
    app_ids=app_ids,
    pop_row=pop_row,
    k_personalization=K_PERSONALIZATION,
)

overall_pers = _append_personalization_metrics(overall.copy(), personalization, on_keys=["method"])
personalization.to_csv(SPIKE_OUT / "v2a_val_personalization.csv", index=False)
overall_pers.to_csv(SPIKE_OUT / "v2a_val_overall_with_personalization.csv", index=False)

display(Markdown("### Personalization"))
display(personalization.sort_values("PersonalizationGapVsPopularity@10", ascending=False))
display(Markdown("### Overall + personalization"))
display(overall_pers.sort_values("NDCG@K", ascending=False))

### Personalization

,method,ILD@10,CatalogCoverage@10,Novelty@10,PersonalizationGapVsPopularity@10
1,two_tower_v1,0.26164,0.98730,12.16756,0.99562
5,two_tower_v1_v2a_query,0.19085,1.00000,9.71895,0.97496
3,two_tower_v1_v2a_history,0.18950,1.00000,9.25637,0.95766
6,two_tower_v1_v2a_query_metadata_logpop_blend,0.20330,0.51746,6.29172,0.72488
4,two_tower_v1_v2a_history_metadata_logpop_blend,0.20366,0.52381,6.27857,0.72388
2,two_tower_v1_heuristic_logpop_blend,0.20463,0.50159,6.27216,0.72010
0,popularity_train,0.21234,0.03492,5.31747,0.00000


### Overall + personalization

,method,Hit@K,Precision@K,Recall@K,MAP@K,NDCG@K,MRR,OracleHit@K,OracleNDCG@K,ILD@10,CatalogCoverage@10,Novelty@10,PersonalizationGapVsPopularity@10
0,two_tower_v1_oracle,0.51224,0.05414,0.49405,0.49405,0.49831,0.51224,0.51224,0.49831,NaN,NaN,NaN,NaN
1,two_tower_v1_v2a_query_metadata_logpop_blend,0.19664,0.01991,0.18743,0.06629,0.09518,0.06911,0.51224,0.49831,0.20330,0.51746,6.29172,0.72488
2,two_tower_v1_v2a_history_metadata_logpop_blend,0.19560,0.01980,0.18669,0.06385,0.09308,0.06652,0.51224,0.49831,0.20366,0.52381,6.27857,0.72388
3,two_tower_v1_heuristic_logpop_blend,0.19328,0.01956,0.18410,0.06432,0.09289,0.06706,0.51224,0.49831,0.20463,0.50159,6.27216,0.72010
4,popularity_train,0.15112,0.01518,0.14680,0.05059,0.07311,0.05222,0.51224,0.49831,0.21234,0.03492,5.31747,0.00000
5,two_tower_v1_v2a_query,0.08552,0.00859,0.07906,0.02995,0.04189,0.03201,0.51224,0.49831,0.19085,1.00000,9.71895,0.97496
6,two_tower_v1_v2a_history,0.08520,0.00855,0.07990,0.02494,0.03812,0.02650,0.51224,0.49831,0.18950,1.00000,9.25637,0.95766
7,two_tower_v1,0.04680,0.00473,0.04374,0.01032,0.01816,0.01101,0.51224,0.49831,0.26164,0.98730,12.16756,0.99562


# Key Findings

**Finding 1:**  
Best retr+meta train_tune: `genre_theme_kw`, **α=0**, Slice A **0.069** — but val retr+meta alone still loses to D1 (Finding 2).

**Finding 2 (retr+meta — kill):**  
`two_tower_v1_v2a_query`: val **0.042** / **0.045** overall / Slice A vs D1 **0.093** / **0.068**.

**Finding 3 (`_metadata_logpop_blend` — promotion candidate):**  
`two_tower_v1_v2a_query_metadata_logpop_blend` (`w_meta=0.05`, `genre_theme_kw`): val **0.095** / **0.070** overall / Slice A — **beats D1** on both; personalization gap **0.725** vs D1 **0.720**. History logpop_blend **0.093** / **0.067** (below query on Slice A).

**Finding 4:**  
Pure metadata failed because it dropped D1’s log-pop prior; small `w_meta` on top of D1 is the lift. Oracle pool NDCG **~0.50** unchanged — rerank headroom captured incrementally.

**Unexpected Results:**  
α=0 retr+meta looked strong on train_tune but val collapsed vs D1; D1+meta at `w_meta=0.05` generalizes.

# Recommendation / Next Steps

**Recommended Action:**  
**Kill** retr+meta-only Jaccard (`two_tower_v1_v2a_query` / `_history`). **Promotion candidate:** `two_tower_v1_v2a_query_metadata_logpop_blend` — passes val promotion bar vs D1. Log in `ranking_decision_log.md`. Head-to-head vs `recs_020` embed logpop_blend before eval-job wiring.

**Risks:**  
Small catalog; FK Jaccard misses sibling tags; embed candidate may edge Slice A further.

**Follow-up:**  
Compare 019 vs 020 logpop_blend winners; wire winner into `recs_job_eval_ranking`; V2b summary logpop_blend if metadata path ships.
